In [ ]:
# Public-release setup: run from any working directory.
from pathlib import Path
import sys

def find_release_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "docs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the public release directory.")

PROJECT_ROOT = find_release_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PROJECT_ROOT / "results"  # User-supplied artifacts; not included in the release.
FIGURES_ROOT = PROJECT_ROOT / "figures"


# Appendix B.2 CH2 Diagnostic Figures

This notebook replots the Appendix B.2 diagnostics from saved CSV/JSON outputs. It does not run model inference or recompute gradients.

The notebook layout is:

1. load all existing diagnostic data;
2. draw the two paper-style combined figures;
3. analyze the single-block restricted-update sweep;
4. expose source tables for caption/debug edits.

Default outputs are written to `figures/appendixB2/` as PNG and PDF.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root whether this notebook is run from repo root or notebook_for_paper/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebook_for_paper" else CWD

RESULTS_ROOT = PROJECT_ROOT / "results"
FIGURE_DIR = PROJECT_ROOT / "figures" / "appendixB2"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

READOUT_DIR = RESULTS_ROOT / "section3_readout_functional_fp32_qwen35_08b_lr1e-4" / "gsm8k_to_dolly" / "Qwen3.5-0.8B" / "section3_validation"
LAST_BLOCK_DIR = RESULTS_ROOT / "last_block_ch2_qwen35_08b_gsm8k_to_dolly_64pairs"
PER_LAYER_DIR = RESULTS_ROOT / "per_layer_ch2_qwen35_08b_gsm8k_to_dolly_64pairs"
SINGLE_BLOCK_DIR = RESULTS_ROOT / "single_block_update_qwen35_08b_gsm8k_to_dolly_64pairs"

PATHS = {
    "readout_pairs": READOUT_DIR / "section3_pairs.csv",
    "readout_diagnostics": READOUT_DIR / "diagnostics.json",
    "last_block_summary": LAST_BLOCK_DIR / "last_block_local_ch2_summary.json",
    "per_layer_summary": PER_LAYER_DIR / "per_layer_ch2_summary.json",
    "per_layer_pairwise": PER_LAYER_DIR / "per_layer_ch2_pairwise.csv",
    "cancellation": PER_LAYER_DIR / "ch2_cancellation_by_pair.csv",
    "single_block_pairs": SINGLE_BLOCK_DIR / "single_block_update_pairs.csv",
    "single_block_summary": SINGLE_BLOCK_DIR / "single_block_update_summary.csv",
    "single_block_run_config": SINGLE_BLOCK_DIR / "run_config.json",
}

for name, path in PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"FIGURE_DIR   = {FIGURE_DIR}")

In [ ]:
# Plot style knobs. Change these and rerun plotting cells.
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLORS = {
    "pearson": "#2f6f9f",
    "spearman_signed": "#c75146",
    "spearman_abs": "#5a8f3d",
    "sign_agreement": "#6f5aa7",
    "exact": "#2f6f9f",
    "approx": "#c75146",
    "actual": "#2f6f9f",
    "reference": "#333333",
    "A_actual_vs_exact": "#4d4d4d",
    "B_exact_vs_approx": "#c75146",
    "C_actual_vs_approx": "#2f6f9f",
}

METRIC_LABELS = {
    "pearson": "Pearson",
    "spearman_signed": "Signed Spearman",
    "spearman_abs": "Abs Spearman",
    "sign_agreement": "Sign agreement",
}

SAVE_PDF = True
SAVE_PNG = True

def save_figure(fig, stem):
    saved = []
    if SAVE_PNG:
        path = FIGURE_DIR / f"{stem}.png"
        fig.savefig(path, bbox_inches="tight")
        saved.append(path)
    if SAVE_PDF:
        path = FIGURE_DIR / f"{stem}.pdf"
        fig.savefig(path, bbox_inches="tight")
        saved.append(path)
    print("saved:", *saved, sep="\n  ")
    return saved

## Load Data

In [ ]:
readout_df = pd.read_csv(PATHS["readout_pairs"])
with open(PATHS["readout_diagnostics"], "r", encoding="utf-8") as f:
    readout_diagnostics = json.load(f)

with open(PATHS["last_block_summary"], "r", encoding="utf-8") as f:
    last_block_summary = json.load(f)
last_block_ladder_df = pd.DataFrame(last_block_summary["approximation_ladder"])

with open(PATHS["per_layer_summary"], "r", encoding="utf-8") as f:
    per_layer_summary_json = json.load(f)
per_layer_summary_df = pd.DataFrame(per_layer_summary_json["per_layer_summary"])
top_down_df = pd.DataFrame(per_layer_summary_json["top_down_cumulative_summary"])
per_layer_pairwise_df = pd.read_csv(PATHS["per_layer_pairwise"])
cancellation_df = pd.read_csv(PATHS["cancellation"])

single_block_summary_df = pd.read_csv(PATHS["single_block_summary"])
single_block_pairs_df = pd.read_csv(PATHS["single_block_pairs"])
with open(PATHS["single_block_run_config"], "r", encoding="utf-8") as f:
    single_block_run_config = json.load(f)

print("readout pairs:", readout_df.shape)
print("last-block ladder:", last_block_ladder_df.shape)
print("per-layer summary:", per_layer_summary_df.shape)
print("top-down cumulative:", top_down_df.shape)
print("cancellation:", cancellation_df.shape)
print("single-block summary:", single_block_summary_df.shape)
print("single-block pairs:", single_block_pairs_df.shape)

## Helper Functions

In [ ]:
def add_zero_lines(ax):
    ax.axhline(0, color="0.82", lw=0.8, zorder=0)
    ax.axvline(0, color="0.82", lw=0.8, zorder=0)


def set_symmetric_limits(ax, x, y, pad=0.08):
    vals = np.concatenate([np.asarray(x, dtype=float), np.asarray(y, dtype=float)])
    lim = np.nanmax(np.abs(vals))
    if not np.isfinite(lim) or lim == 0:
        lim = 1.0
    lim *= 1 + pad
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    return lim


def clean_comparison_label(label):
    labels = {
        "exact_block_vs_sequence_factorized_linear": "exact block\nvs factors",
        "linear_direct_vs_sequence_factorized_linear": "linear direct\nvs factors",
        "linear_direct_vs_target_position_factorized": "target-pos\nfactors",
        "linear_direct_vs_q_readout_force_only": "readout-force\nonly",
        "linear_direct_vs_exact_force_same_inputs": "exact force\nsame inputs",
        "linear_direct_vs_current_ch2_last_block": "current CH2\nlast block",
        "linear_direct_vs_last_attn_norm_force_only": "attn-norm\nforce only",
    }
    return labels.get(label, label.replace("_", "\n"))


def plot_readout_parity_panel(ax, df=readout_df):
    x = df["first_order_exact_readout_functional"].to_numpy(dtype=float)
    y = df["functional_readout_sanity_target"].to_numpy(dtype=float)
    metrics = readout_diagnostics["comparisons"]["functional_readout_first_order_vs_lr_ch1"]
    ax.scatter(x, y, s=28, alpha=0.8, color=COLORS["pearson"], edgecolor="white", linewidth=0.4)
    lim = set_symmetric_limits(ax, x, y)
    ax.plot([-lim, lim], [-lim, lim], color=COLORS["reference"], lw=1.0, ls="--")
    add_zero_lines(ax)
    ax.set_xlabel("Exact functional readout")
    ax.set_ylabel(r"$lr \cdot CH1$")
    ax.set_title("CH1 readout sanity")
    ax.text(
        0.04, 0.96,
        f"Pearson={metrics['pearson']:.6f}\nSign={metrics['sign_agreement']:.3f}\nMAE={metrics['mae']:.1e}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8,
        bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25"},
    )


def plot_last_block_ladder_panel(ax, df=last_block_ladder_df):
    plot_df = df.copy()
    plot_df["label"] = plot_df["comparison"].map(clean_comparison_label)
    x = np.arange(len(plot_df))
    width = 0.24
    ax.bar(x - width, plot_df["pearson"], width, label="Pearson", color=COLORS["pearson"])
    ax.bar(x, plot_df["spearman_signed"], width, label="Signed Spearman", color=COLORS["spearman_signed"])
    ax.bar(x + width, plot_df["sign_agreement"], width, label="Sign agreement", color=COLORS["sign_agreement"])
    ax.axhline(0, color="0.75", lw=0.8)
    ax.axhline(1, color="0.86", lw=0.8, ls="--")
    ax.set_ylim(-0.25, 1.08)
    ax.set_ylabel("Metric")
    ax.set_title("Last-block CH2 ladder")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["label"], rotation=0, ha="center")
    ax.legend(frameon=False, ncol=3, loc="lower left")


def plot_signed_quality_panel(ax, df=per_layer_summary_df):
    for key in ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]:
        ax.plot(df["layer"], df[key], marker="o", ms=3, lw=1.4, label=METRIC_LABELS[key], color=COLORS[key])
    ax.axhline(0, color="0.78", lw=0.8)
    ax.axhline(0.5, color="0.86", lw=0.8, ls="--")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Metric")
    ax.set_ylim(-0.7, 1.05)
    ax.set_title("Layer-wise CH2 quality")
    ax.legend(frameon=False, fontsize=8)


def plot_top_down_panel(ax, df=top_down_df):
    x = df["top_k_layers"]
    for key in ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]:
        ax.plot(x, df[key], marker="o", ms=3, lw=1.4, label=METRIC_LABELS[key], color=COLORS[key])
    ax.axhline(0, color="0.78", lw=0.8)
    ax.axhline(0.5, color="0.86", lw=0.8, ls="--")
    ax.set_xlabel("Included top layers")
    ax.set_ylabel("Metric")
    ax.set_ylim(-0.35, 1.05)
    ax.set_title("Top-down cumulative CH2")

## Combined Figure 1: CH1 Sanity and Last-Block CH2 Ladder

In [ ]:
def plot_ch1_and_ladder():
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1.0, 3]})
    plot_readout_parity_panel(axes[0])
    plot_last_block_ladder_panel(axes[1])
    axes[0].text(-0.18, 1.05, "a", transform=axes[0].transAxes, fontweight="bold", fontsize=12)
    axes[1].text(-0.08, 1.05, "b", transform=axes[1].transAxes, fontweight="bold", fontsize=12)
    fig.tight_layout()
    save_figure(fig, "appendixB2_ch1_sanity_and_last_block_ladder")
    return fig, axes

plot_ch1_and_ladder();

## Combined Figure 2: Layer-Wise and Cumulative CH2 Quality

In [ ]:
def plot_layer_and_cumulative_quality():
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharey=False)
    plot_signed_quality_panel(axes[0])
    plot_top_down_panel(axes[1])
    axes[0].text(-0.14, 1.05, "a", transform=axes[0].transAxes, fontweight="bold", fontsize=12)
    axes[1].text(-0.14, 1.05, "b", transform=axes[1].transAxes, fontweight="bold", fontsize=12)
    fig.tight_layout()
    save_figure(fig, "appendixB2_layerwise_and_cumulative_ch2")
    return fig, axes

plot_layer_and_cumulative_quality();

## Optional Figure: Cross-Layer Cancellation

In [ ]:
def plot_cancellation(df=cancellation_df):
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))

    bins = np.linspace(0, 1.05, 22)
    axes[0].hist(df["exact_cancellation_ratio"], bins=bins, alpha=0.72, label="Exact", color=COLORS["exact"])
    axes[0].hist(df["approx_cancellation_ratio"], bins=bins, alpha=0.55, label="Approx", color=COLORS["approx"])
    axes[0].set_xlabel(r"$|\sum_l c_l| / \sum_l |c_l|$")
    axes[0].set_ylabel("Pairs")
    axes[0].set_title("Cancellation ratio")
    axes[0].legend(frameon=False)

    correct = df[df["sum_sign_correct"].astype(bool)]
    wrong = df[~df["sum_sign_correct"].astype(bool)]
    axes[1].scatter(correct["exact_cancellation_ratio"], correct["approx_cancellation_ratio"], s=30, alpha=0.75, color=COLORS["exact"], label="Sign correct", edgecolor="white", linewidth=0.4)
    axes[1].scatter(wrong["exact_cancellation_ratio"], wrong["approx_cancellation_ratio"], s=30, alpha=0.75, color=COLORS["approx"], label="Sign wrong", edgecolor="white", linewidth=0.4)
    axes[1].plot([0, 1], [0, 1], color=COLORS["reference"], lw=1.0, ls="--")
    axes[1].set_xlim(0, 1.05)
    axes[1].set_ylim(0, 1.05)
    axes[1].set_xlabel("Exact cancellation ratio")
    axes[1].set_ylabel("Approx cancellation ratio")
    axes[1].set_title("Approx misses exact cancellation")
    axes[1].legend(frameon=False)

    fig.tight_layout()
    save_figure(fig, "appendixB2_cancellation")
    return fig, axes

plot_cancellation();

## Single-Block Restricted-Update Sweep

This section analyzes the new true single-block update experiment. For each layer, the runner froze all other parameters, restored the base model before each unique update token, applied one SGD step on that layer only, and measured each paired observation token.

Comparisons:

- `A_actual_vs_exact`: actual block-only `delta_logp_layer` vs `first_order_exact_layer_l`.
- `B_exact_vs_approx`: `first_order_exact_layer_l` vs `lr * ch2_layer_l`.
- `C_actual_vs_approx`: actual block-only `delta_logp_layer` vs `lr * ch2_layer_l`.

In [ ]:
# Quick numeric summary for the single-block run.
sb_c = single_block_summary_df[single_block_summary_df["comparison"] == "C_actual_vs_approx"].copy()
sb_a = single_block_summary_df[single_block_summary_df["comparison"] == "A_actual_vs_exact"].copy()
sb_b = single_block_summary_df[single_block_summary_df["comparison"] == "B_exact_vs_approx"].copy()

final_layer = int(sb_c["layer"].max())
representative_layers = [0, 12, final_layer]
print("single-block run:", single_block_run_config["num_layers"], "layers,", single_block_run_config["num_pairs"], "pairs")
print("LR:", single_block_run_config["lr"], "dtype:", single_block_run_config["torch_dtype"])
print("C actual-vs-CH2 selected layers")
display(sb_c[sb_c["layer"].isin(representative_layers)])
print("Worst C sign-agreement layers")
display(sb_c.sort_values("sign_agreement").head(6))

In [ ]:
def plot_single_block_metrics(summary=single_block_summary_df):
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4), sharex=True)

    c = summary[summary["comparison"] == "C_actual_vs_approx"]
    ax = axes[0]
    for key in ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]:
        ax.plot(c["layer"], c[key], marker="o", ms=3, lw=1.4, label=METRIC_LABELS[key], color=COLORS[key])
    ax.axhline(0, color="0.78", lw=0.8)
    ax.axhline(0.5, color="0.86", lw=0.8, ls="--")
    ax.set_ylim(-0.75, 1.05)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Metric")
    ax.set_title("Actual block update vs CH2_l")
    ax.legend(frameon=False, fontsize=8)

    ax = axes[1]
    for comparison, label, style in [
        ("A_actual_vs_exact", "Actual vs exact", "--"),
        ("B_exact_vs_approx", "Exact vs CH2_l", ":"),
        ("C_actual_vs_approx", "Actual vs CH2_l", "-"),
    ]:
        ss = summary[summary["comparison"] == comparison]
        ax.plot(ss["layer"], ss["sign_agreement"], marker="o", ms=3, lw=1.4, linestyle=style, label=label, color=COLORS[comparison])
    ax.axhline(0.5, color="0.86", lw=0.8, ls="--")
    ax.set_ylim(0.25, 1.05)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Sign agreement")
    ax.set_title("Taylor vs structural error")
    ax.legend(frameon=False, fontsize=8)

    axes[0].text(-0.14, 1.05, "a", transform=axes[0].transAxes, fontweight="bold", fontsize=12)
    axes[1].text(-0.14, 1.05, "b", transform=axes[1].transAxes, fontweight="bold", fontsize=12)
    fig.tight_layout()
    save_figure(fig, "appendixB2_single_block_update_metrics")
    return fig, axes

plot_single_block_metrics();

In [ ]:
def plot_single_block_representative_scatters(pair_df=single_block_pairs_df, layers=None):
    if layers is None:
        layers = [0, 12, int(pair_df["layer"].max())]
    n = len(layers)
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.2), squeeze=False)
    axes = axes.ravel()
    for ax, layer in zip(axes, layers):
        sub = pair_df[pair_df["layer"] == layer]
        x = sub["delta_logp_layer"].to_numpy(dtype=float)
        y = sub["approx_layer"].to_numpy(dtype=float)
        ax.scatter(x, y, s=22, alpha=0.75, color=COLORS["actual"], edgecolor="white", linewidth=0.35)
        lim = set_symmetric_limits(ax, x, y, pad=0.08)
        ax.plot([-lim, lim], [-lim, lim], color=COLORS["reference"], lw=0.9, ls="--")
        add_zero_lines(ax)
        metrics = sb_c[sb_c["layer"] == layer].iloc[0]
        ax.set_title(f"Layer {layer}\nPearson={metrics['pearson']:.2f}, sign={metrics['sign_agreement']:.2f}")
        ax.set_xlabel("Actual delta_logp")
        ax.set_ylabel(r"$lr \cdot CH2_l$")
    fig.tight_layout()
    save_figure(fig, "appendixB2_single_block_representative_scatters")
    return fig, axes

plot_single_block_representative_scatters(layers=[0, 12, 23]);

## Source Tables

In [ ]:
# These are useful when editing captions or deciding which metrics to show.
display(readout_diagnostics["comparisons"]["functional_readout_first_order_vs_lr_ch1"])
display(last_block_ladder_df)
display(per_layer_summary_df)
display(top_down_df)
display(cancellation_df.describe())
display(single_block_summary_df)